# Predictive Workforce Analytics — Graduate Pay & Placement Intelligence
### End-to-end Big Data pipeline · Hadoop ecosystem → Spark ML

**Course** Big Data Analytics · MBA (Business Analytics), Trimester IV
**Dataset** NUS Graduate Employment Survey 2024, Ministry of Education Singapore
**Platform** Apache Spark / Databricks

---

## Coverage map — how this notebook maps to the syllabus

| Session | Topic | Where it appears |
|---|---|---|
| 1–2 | Big Data concepts, Hadoop ecosystem, HDFS | Part 1 |
| 3 | HDFS commands | Part 2 |
| 4–5 | MapReduce programming logic | Part 3 |
| 6 | Apache Pig | Part 4 |
| 7 | Apache Hive — partitions, buckets, SerDe | Part 5 |
| 8 | Apache Sqoop — RDBMS ingestion | Part 6 |
| 9 | HBase, Kafka, Cassandra | Part 7 |
| 10, 12 | Spark architecture, RDD, DAG, DataFrames | Part 8 |
| 13 | Feature engineering | Part 9 |
| 14–15 | Regression | Part 10 |
| 16–17 | Classification | Part 11 |
| 18–19 | Clustering | Part 12 |

---

## The business problem

Employers set fresh-graduate pay on instinct. A single company-wide "graduate band" is applied
across degrees whose market rates differ by 2×, and CVs are screened on GPA cut-offs never
validated against outcomes. This notebook builds the data pipeline and models that replace
the guess — and tests whether the resulting models actually hold up.

## Part 0 · Environment setup

In [ ]:
# Works on Databricks, Google Colab, or local Spark
import sys, os, subprocess

ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    subprocess.run(["pip","install","-q","pyspark"], check=False)
    print("PySpark installed on Colab")

print("Databricks:", ON_DATABRICKS, "| Colab:", ON_COLAB)

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
import math, random

if ON_DATABRICKS:
    pass  # 'spark' is pre-created
else:
    spark = (SparkSession.builder
             .appName("GraduateWorkforceAnalytics")
             .master("local[*]")
             .config("spark.sql.shuffle.partitions", "8")
             .config("spark.sql.warehouse.dir", "/tmp/hive_warehouse")
             .config("spark.driver.memory", "4g")
             .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

print("Spark version :", spark.version)
print("Master        :", sc.master)
print("Default parallelism (cores):", sc.defaultParallelism)

## Part 1 · The Hadoop ecosystem and why we need it

Before any code: **what problem does this stack solve?**

A single machine reads roughly 100 MB/s from disk. Reading 1 TB takes about 3 hours.
Split that file across 100 machines and it takes under 2 minutes. That is the entire
idea — **move the computation to the data, not the data to the computation.**

| Layer | Tool | Job |
|---|---|---|
| Storage | **HDFS** | Split files into blocks, replicate 3×, survive machine failure |
| Resource management | **YARN** | Decide which machine runs which task |
| Batch processing | **MapReduce** | Process blocks in parallel, write intermediate results to disk |
| Ingestion — bulk | **Sqoop** | Move data between RDBMS and HDFS |
| Ingestion — stream | **Flume / Kafka** | Continuous event capture |
| SQL on files | **Hive** | Schema-on-read, SQL over HDFS files |
| Scripting | **Pig** | Dataflow scripting, compiles to MapReduce |
| NoSQL | **HBase / Cassandra** | Random real-time read/write on large tables |
| In-memory engine | **Spark** | Same idea as MapReduce, but keeps data in RAM — 10–100× faster |

Our dataset is deliberately small enough to inspect, but every operation below is written
the way it would be written at terabyte scale.

In [ ]:
# HDFS block arithmetic — why block size matters
FILE_SIZE_GB   = 1024      # a 1 TB survey archive
BLOCK_SIZE_MB  = 128       # HDFS default
REPLICATION    = 3

blocks = (FILE_SIZE_GB * 1024) // BLOCK_SIZE_MB
print(f"File size            : {FILE_SIZE_GB} GB")
print(f"HDFS block size      : {BLOCK_SIZE_MB} MB")
print(f"Blocks created       : {blocks:,}")
print(f"Replication factor   : {REPLICATION}")
print(f"Raw storage consumed : {FILE_SIZE_GB * REPLICATION:,} GB")
print(f"Max parallel readers : {blocks:,} (one task per block)")
print()
print(f"Single machine @100MB/s : {FILE_SIZE_GB*1024/100/60:,.0f} min")
print(f"100-node cluster        : {FILE_SIZE_GB*1024/100/60/100:,.1f} min")

## Part 2 · HDFS — storage layer

On a real cluster the data lands in HDFS. On Databricks the equivalent is **DBFS**.
The commands below are the ones you would run; we then create the equivalent local
directory structure so the rest of the notebook is runnable anywhere.

```bash
# Inspect the filesystem
hdfs dfs -ls /user/analytics/graduates
hdfs dfs -mkdir -p /user/analytics/graduates/raw

# Load a local file into HDFS
hdfs dfs -put ges_2024.csv /user/analytics/graduates/raw/

# Check how the file was split and where blocks live
hdfs fsck /user/analytics/graduates/raw/ges_2024.csv -files -blocks -locations

# Storage report
hdfs dfsadmin -report
```

In [ ]:
# Create the working layout (DBFS on Databricks, local dirs otherwise)
BASE = "/tmp/graduates" if not ON_DATABRICKS else "/dbfs/tmp/graduates"
SPARK_BASE = BASE if not ON_DATABRICKS else "dbfs:/tmp/graduates"

for sub in ["raw", "warehouse", "curated", "stream", "checkpoint"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)

print("Storage layout created under:", BASE)
for sub in sorted(os.listdir(BASE)):
    print("  /" + sub)

# On Databricks you would use dbutils instead:
#   dbutils.fs.mkdirs("dbfs:/tmp/graduates/raw")
#   dbutils.fs.ls("dbfs:/tmp/graduates")

## Part 3 · Source data and MapReduce

### 3.1 Reconstructing the population

The MOE survey publishes **summary statistics per programme**, not individual records —
privacy legislation restricts the microdata. To model at individual level we reconstruct
the population by sampling from distributions fitted to each programme's published
25th, 50th and 75th percentiles.

This is standard practice in workforce analytics. The **aggregate statistics are authentic**;
individual observations are simulated to preserve them.

In [ ]:
# NUS GES 2024 — published summary statistics
# (faculty, degree, employed_rate, ft_perm_rate, gross_mean, gross_median, p25, p75, honours, cohort)
GES_2024 = [
 ("Arts & Social Sciences","Bachelor of Arts",        .935,.742,5474,4300,3600,6250,0, 80),
 ("Arts & Social Sciences","BA (Hons)",               .830,.674,4476,4419,4000,4830,1,200),
 ("Arts & Social Sciences","Social Sciences",         .861,.739,4462,4210,3820,4800,0,250),
 ("Law",                   "Bachelor of Laws",        .938,.904,6999,7000,6200,7000,1,180),
 ("Science",               "BSc (Hons)",              .853,.754,4469,4150,3900,4800,1,300),
 ("Science",               "Data Science & Analytics",.859,.770,5855,5400,4800,6600,1,130),
 ("Science",               "Computational Biology",   .833,.750,5252,5240,4200,5558,1, 30),
 ("Business",              "BBA",                     .855,.818,6903,5100,4200,8500,0,200),
 ("Business",              "BBA (Hons)",              .911,.849,5430,4900,4200,5800,1,350),
 ("Business",              "Accountancy (Hons)",      .969,.945,4864,4388,4250,5242,1,250),
 ("Business",              "Real Estate",             .925,.817,4452,4100,3800,4600,0,120),
 ("Computing",             "Computer Science",        .891,.878,6788,6500,5600,7500,1,400),
 ("Computing",             "Information Security",    .912,.882,6581,6110,5500,7049,1, 80),
 ("Computing",             "Information Systems",     .903,.875,6132,6000,5200,6955,1,150),
 ("Computing",             "Business Analytics",      .925,.878,5712,5400,4900,6350,1,160),
 ("Engineering",           "Biomedical Eng",          .702,.638,4482,4325,4000,4893,1,100),
 ("Engineering",           "Chemical Eng",            .901,.874,4892,4760,4300,5000,1,150),
 ("Engineering",           "Civil Eng",               .942,.884,4420,4200,4000,4935,1,130),
 ("Engineering",           "Electrical Eng",          .884,.860,5057,5000,4500,5400,1,100),
 ("Engineering",           "Mechanical Eng",          .853,.824,5038,5000,4380,5500,1,150),
 ("Engineering",           "Industrial & Systems",    .857,.821,4905,4900,4300,5500,1, 80),
 ("Engineering",           "Computer Eng",            .904,.842,5896,5800,5000,6666,1,120),
 ("Engineering",           "Industrial Design",       .767,.600,4111,4025,3500,4500,0, 60),
 ("Engineering",           "Project & Facilities",    .864,.807,4018,4000,3800,4290,1, 90),
 ("Medicine",              "Nursing",                 .883,.834,3993,3950,3650,4110,0,120),
 ("Medicine",              "Nursing (Hons)",          .949,.943,4074,4050,3880,4250,1,180),
 ("Music",                 "Bachelor of Music",       .889,.278,3820,3700,3600,4100,0, 40),
 ("Yale-NUS",              "BA (Hons) YNC",           .844,.733,5147,4557,4200,5950,1, 50),
 ("Yale-NUS",              "BSc (Hons) YNC",          .892,.811,5898,5043,4500,6400,1, 40),
 ("Multi-Disciplinary",    "Environmental Studies",   .900,.625,4354,4300,4000,4500,0, 50),
]
print(f"{len(GES_2024)} programmes | published cohort {sum(r[9] for r in GES_2024):,}")

In [ ]:
# Generate individual records with an explicit causal structure:
#   degree sets the salary BAND; GPA and internships shift position WITHIN it.
import numpy as np
np.random.seed(42)
random.seed(42)

def field_of(fac, deg):
    if fac == "Computing" or "Computer" in deg or "Data Sci" in deg or "Business Analytics" in deg:
        return "Computing"
    return {"Engineering":"Engineering","Science":"Science","Business":"Business",
            "Medicine":"Health","Law":"Law"}.get(fac, "Arts")

def lognorm_params(p25, med, p75):
    mu = math.log(max(med, 1))
    sigma = (math.log(p75) - math.log(p25)) / 1.35 if p75 > p25 > 0 else 0.10
    return mu, max(sigma, 0.05)

records, sid = [], 1
for fac, deg, emp, ftp, gmean, gmed, p25, p75, hons, n in GES_2024:
    fc = field_of(fac, deg)
    mu, sigma = lognorm_params(p25, gmed, p75)
    for _ in range(n):
        gpa = float(np.clip(np.random.normal(3.5, 0.42), 2.0, 5.0))
        interns = int(min(np.random.poisson(2.2 if fc in ("Business","Computing","Law")
                                            else 1.5), 6))
        boost = (gpa - 3.5) * 0.10 + (interns - 2) * 0.035
        employed = 1 if np.random.rand() < np.clip(emp + boost, .05, .99) else 0
        ftperm = 0
        if employed:
            cond = min(ftp / emp, 1.0) if emp > 0 else 0
            ftperm = 1 if np.random.rand() < np.clip(cond + boost, .05, .99) else 0
        if ftperm:
            base = float(np.random.lognormal(mu, sigma * 0.72))
            lift = 1 + (gpa-3.5)*0.055 + (interns-2)*0.022 + 0.02*hons
            gross = float(min(max(base * lift, 2000.0), 15000.0))
        else:
            gross = 0.0
        records.append((sid, fac, deg, fc, bool(hons), round(gpa,2), interns,
                        employed, ftperm, float(round(gross,0))))
        sid += 1

print(f"Generated {len(records):,} individual graduate records")

In [ ]:
SCHEMA = StructType([
    StructField("student_id",      IntegerType(),  False),
    StructField("faculty",         StringType(),   False),
    StructField("degree",          StringType(),   False),
    StructField("field",           StringType(),   False),
    StructField("honours",         BooleanType(),  False),
    StructField("gpa",             DoubleType(),   False),
    StructField("internships",     IntegerType(),  False),
    StructField("employed",        IntegerType(),  False),
    StructField("ft_permanent",    IntegerType(),  False),
    StructField("gross_salary",    DoubleType(),   False),
])

grads = spark.createDataFrame(records, schema=SCHEMA).cache()
print("Records :", f"{grads.count():,}")
print("Partitions:", grads.rdd.getNumPartitions())
grads.printSchema()
grads.show(5, truncate=False)

In [ ]:
# Land it in the 'HDFS' raw zone as CSV — this is what Flume/Sqoop would deposit
(grads.coalesce(1).write.mode("overwrite")
      .option("header", True)
      .csv(f"file://{BASE}/raw/graduates_csv" if not ON_DATABRICKS
           else f"{SPARK_BASE}/raw/graduates_csv"))
print("Raw CSV written to HDFS raw zone")

### 3.2 MapReduce — the original processing model

MapReduce is two functions and a shuffle between them:

```
INPUT → map(k1,v1) → [(k2,v2)] → SHUFFLE & SORT → reduce(k2,[v2]) → OUTPUT
```

The classic word-count, expressed against our data: *count graduates per faculty.*
The **mapper** emits `(faculty, 1)` for every record. The **shuffle** groups identical
keys onto the same reducer. The **reducer** sums each group.

Spark's RDD API exposes exactly this, so we can run genuine MapReduce below.

In [ ]:
# ---- Genuine MapReduce on RDDs ----
rdd = grads.rdd

# MAP phase — emit (key, value) pairs
mapped = rdd.map(lambda r: (r["faculty"], 1))

# SHUFFLE & REDUCE phase
reduced = mapped.reduceByKey(lambda a, b: a + b)

print("MapReduce — graduates per faculty")
print("-" * 44)
for fac, cnt in sorted(reduced.collect(), key=lambda x: -x[1]):
    print(f"  {fac:<26} {cnt:>6,}")

In [ ]:
# A second MapReduce: average salary per field.
# The reducer must carry (sum, count) so the average can be computed at the end.
salaried = rdd.filter(lambda r: r["ft_permanent"] == 1)

avg_by_field = (salaried
    .map(lambda r: (r["field"], (r["gross_salary"], 1)))          # MAP
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))             # REDUCE
    .mapValues(lambda x: x[0] / x[1]))                            # finalise

print("MapReduce — mean gross salary by field")
print("-" * 44)
for fld, avg in sorted(avg_by_field.collect(), key=lambda x: -x[1]):
    print(f"  {fld:<16} S${avg:>9,.0f}")

**Why MapReduce was replaced.** Every stage writes its intermediate result to disk before
the next stage reads it. A chain of ten operations means ten round-trips to disk. Spark keeps
intermediate results in memory and only spills when it must — the same logical model,
one to two orders of magnitude faster.

The equivalent Java mapper for reference:

```java
public class FacultyMapper extends Mapper<LongWritable, Text, Text, IntWritable> {
    private final IntWritable ONE = new IntWritable(1);
    public void map(LongWritable key, Text value, Context ctx)
            throws IOException, InterruptedException {
        String faculty = value.toString().split(",")[1];
        ctx.write(new Text(faculty), ONE);
    }
}
```

## Part 4 · Apache Pig — dataflow scripting

Pig Latin describes a **dataflow** rather than a query. It compiles down to MapReduce jobs.
Its value was letting analysts express multi-step transformations without writing Java.

The Pig script for our salary-by-faculty aggregation:

```pig
-- Load with an explicit schema (schema-on-read)
grads = LOAD '/user/analytics/graduates/raw/graduates.csv'
        USING PigStorage(',')
        AS (student_id:int, faculty:chararray, degree:chararray, field:chararray,
            honours:boolean, gpa:double, internships:int,
            employed:int, ft_permanent:int, gross_salary:double);

-- The four verbs: FILTER, GROUP, aggregate (TOTAL), ORDER (SORT)
salaried  = FILTER grads BY ft_permanent == 1;
by_fac    = GROUP salaried BY faculty;
summary   = FOREACH by_fac GENERATE
                group AS faculty,
                COUNT(salaried) AS n,
                AVG(salaried.gross_salary) AS mean_salary;
ranked    = ORDER summary BY mean_salary DESC;

STORE ranked INTO '/user/analytics/graduates/pig_out' USING PigStorage(',');
DUMP ranked;   -- nothing executes until DUMP or STORE (lazy evaluation)
```

Note `DUMP` at the end — **Pig is lazy**, exactly like Spark. The script builds a plan;
nothing runs until an action requests output. Below is the identical logic in Spark.

In [ ]:
# The Pig script above, expressed in Spark — same four verbs
pig_equivalent = (grads
    .filter(F.col("ft_permanent") == 1)                    # FILTER
    .groupBy("faculty")                                     # GROUP
    .agg(F.count("*").alias("n"),                           # TOTAL
         F.round(F.avg("gross_salary"), 0).alias("mean_salary"))
    .orderBy(F.desc("mean_salary")))                        # ORDER

pig_equivalent.show(truncate=False)

## Part 5 · Apache Hive — SQL over files

Hive's central idea is **schema-on-read**. The data already sits in HDFS; Hive stores only
a *description* of its shape in the **metastore**. You are labelling a filing cabinet you
already own, not loading data into a database.

Two physical layouts matter:

- **Partitioning** — one directory per value (`/faculty=Computing/`). A query filtered on
  faculty reads only that directory. This is *partition pruning*.
- **Bucketing** — hash rows into a fixed number of files inside each partition. Makes joins
  and sampling cheaper because matching keys land in the same bucket.

Spark SQL is Hive-compatible, so the DDL below is real and runnable.

In [ ]:
# Drop cleanly so the notebook is safely re-runnable
spark.sql("DROP DATABASE IF EXISTS graduate_analytics CASCADE")
if not ON_DATABRICKS:
    subprocess.run(["rm","-rf","/tmp/hive_warehouse/graduate_analytics.db"])

spark.sql("CREATE DATABASE IF NOT EXISTS graduate_analytics")
spark.sql("USE graduate_analytics")

grads.write.mode("overwrite").saveAsTable("graduates_raw")

print("Databases:")
spark.sql("SHOW DATABASES").show(truncate=False)
print("Tables in graduate_analytics:")
spark.sql("SHOW TABLES").show(truncate=False)

In [ ]:
# The equivalent Hive DDL for an external, partitioned table
hive_ddl = '''
CREATE EXTERNAL TABLE IF NOT EXISTS graduates (
    student_id    INT,
    degree        STRING,
    field         STRING,
    honours       BOOLEAN,
    gpa           DOUBLE,
    internships   INT,
    employed      INT,
    ft_permanent  INT,
    gross_salary  DOUBLE
)
PARTITIONED BY (faculty STRING)
CLUSTERED BY (degree) INTO 8 BUCKETS
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS PARQUET
LOCATION '/user/analytics/graduates/warehouse';

-- Let Hive discover partitions already on disk
MSCK REPAIR TABLE graduates;
SET hive.exec.dynamic.partition.mode=nonstrict;
'''
print(hive_ddl)

In [ ]:
# Write a genuinely partitioned + bucketed table
spark.sql("DROP TABLE IF EXISTS graduates_partitioned")
if not ON_DATABRICKS:
    subprocess.run(["rm","-rf","/tmp/hive_warehouse/graduate_analytics.db/graduates_partitioned"])

(grads.write
      .mode("overwrite")
      .partitionBy("faculty")
      .bucketBy(8, "degree")
      .sortBy("degree")
      .format("parquet")
      .saveAsTable("graduates_partitioned"))

print("Partitioned + bucketed table written.\n")
spark.sql("SHOW PARTITIONS graduates_partitioned").show(12, truncate=False)

In [ ]:
# Partition pruning in action — read the physical plan
q = spark.sql('''
    SELECT degree,
           COUNT(*)                        AS graduates,
           ROUND(AVG(gross_salary), 0)     AS mean_salary
    FROM graduates_partitioned
    WHERE faculty = 'Computing' AND ft_permanent = 1
    GROUP BY degree
    ORDER BY mean_salary DESC
''')
q.show(truncate=False)

print("\n--- Physical plan (look for PartitionFilters) ---")
q.explain(mode="formatted")

### SerDe — the customs officer

A **SerDe** (Serializer/Deserializer) tells Hive how to unpack each record. For structured
CSV the default works. For semi-structured data such as web server logs you supply a regex:

```sql
CREATE EXTERNAL TABLE apache_logs (
    host STRING, identity STRING, user STRING, time STRING,
    request STRING, status STRING, size STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.RegexSerDe'
WITH SERDEPROPERTIES (
  "input.regex" = "([^ ]*) ([^ ]*) ([^ ]*) (-|\\[[^\\]]*\\]) ([^ \"]*|\"[^\"]*\") (-|[0-9]*) (-|[0-9]*)"
)
LOCATION '/user/analytics/logs';
```

The file on disk never changes. The SerDe imposes a shape on it at read time.

## Part 6 · Apache Sqoop — bulk RDBMS transfer

University records live in a relational database (registrar systems, HR systems). Sqoop
moves data **between an RDBMS and HDFS in bulk**, in both directions, on a schedule.

Under the hood Sqoop is a MapReduce job: it splits the table by a numeric key into ranges,
then each mapper pulls its own range over JDBC in parallel.

```bash
# IMPORT — RDBMS → HDFS, as Hive table, split across 4 parallel mappers
sqoop import \
  --connect jdbc:mysql://registrar.nus.edu.sg:3306/student_records \
  --username analytics --password-file /user/analytics/.pw \
  --table graduate_outcomes \
  --split-by student_id --num-mappers 4 \
  --hive-import --hive-database graduate_analytics \
  --hive-table graduates_from_rdbms \
  --target-dir /user/analytics/graduates/sqoop_import

# INCREMENTAL import — only rows newer than the last watermark
sqoop import ... \
  --incremental append --check-column student_id --last-value 4340

# EXPORT — push model scores back to the operational database
sqoop export \
  --connect jdbc:mysql://registrar.nus.edu.sg:3306/student_records \
  --table salary_predictions \
  --export-dir /user/analytics/graduates/curated/predictions
```

We demonstrate the same movement below using a real SQLite database and Spark's JDBC
reader — which is precisely what each Sqoop mapper does.

In [ ]:
import sqlite3, pandas as pd

# ---- Build a real relational source database ----
DB = f"{BASE}/registrar.db"
if os.path.exists(DB): os.remove(DB)

con = sqlite3.connect(DB)
pdf = pd.DataFrame(records, columns=[f.name for f in SCHEMA.fields])
pdf.to_sql("graduate_outcomes", con, index=False)

# A second table to join against — faculty reference data
fac_ref = pd.DataFrame(
    [(f, sum(1 for r in GES_2024 if r[0]==f)) for f in sorted({r[0] for r in GES_2024})],
    columns=["faculty","n_programmes"])
fac_ref.to_sql("faculty_reference", con, index=False)

con.commit()
print("Source RDBMS created:", DB)
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
print()
print("Row count:", pd.read_sql("SELECT COUNT(*) n FROM graduate_outcomes", con).iloc[0,0])
con.close()

In [ ]:
# ---- The Sqoop-equivalent import via JDBC ----
# Sqoop splits on a numeric column; Spark does the same with partitionColumn.
try:
    jdbc_ok = True
    imported = (spark.read.format("jdbc")
        .option("url", f"jdbc:sqlite:{DB}")
        .option("dbtable", "graduate_outcomes")
        .option("driver", "org.sqlite.JDBC")
        .load())
    print("JDBC import rows:", imported.count())
except Exception as e:
    jdbc_ok = False
    print("SQLite JDBC driver not on the classpath in this environment.")
    print("On a real cluster Sqoop supplies the driver. Falling back to a")
    print("direct read to demonstrate the same split-by-key logic.\n")

    # Demonstrate the SPLIT logic Sqoop uses, explicitly
    NUM_MAPPERS = 4
    lo, hi = 1, len(records)
    stride = (hi - lo + 1) // NUM_MAPPERS
    print(f"--split-by student_id --num-mappers {NUM_MAPPERS}")
    print(f"Sqoop would issue {NUM_MAPPERS} parallel queries:\n")
    con = sqlite3.connect(DB)
    parts = []
    for m in range(NUM_MAPPERS):
        a = lo + m*stride
        b = hi if m == NUM_MAPPERS-1 else a + stride - 1
        sql = f"SELECT * FROM graduate_outcomes WHERE student_id >= {a} AND student_id <= {b}"
        print(f"  mapper {m}: {sql}")
        parts.append(pd.read_sql(sql, con))
    con.close()
    imported = spark.createDataFrame(pd.concat(parts, ignore_index=True))
    print(f"\nImported {imported.count():,} rows across {NUM_MAPPERS} simulated mappers")

imported.show(3, truncate=False)

In [ ]:
# Land the imported data in the curated zone as Parquet (columnar, compressed)
out = (f"file://{BASE}/curated/graduates_parquet" if not ON_DATABRICKS
       else f"{SPARK_BASE}/curated/graduates_parquet")
imported.write.mode("overwrite").parquet(out)
print("Sqoop import landed in curated zone as Parquet")
print("Files written:", len([f for f in os.listdir(f'{BASE}/curated/graduates_parquet')
                             if f.endswith('.parquet')]))

## Part 7 · Kafka, Cassandra and HBase — the real-time tier

Batch tools answer *"what happened last quarter?"*. The tools below answer
*"what is happening right now?"* and *"fetch this one record in 5 milliseconds."*

### Kafka — the conveyor belt
A distributed, replayable log. Producers append events; multiple independent consumer
groups read the same stream at their own pace. Retention is time-based, not
delete-on-read — you can replay history.

In our context: graduate placement events arriving continuously as offers are accepted.

In [ ]:
# ---- Kafka producer/consumer, as it would be written ----
kafka_code = '''
# PRODUCER — a university career office publishing placement events
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers=["broker1:9092","broker2:9092"],
    value_serializer=lambda v: json.dumps(v).encode())

producer.send("placement-events", {
    "student_id": 4821, "degree": "Computer Science",
    "employer": "Acme Tech", "gross_salary": 6800,
    "event_time": "2026-08-08T10:14:22Z"
})

# CONSUMER — Spark Structured Streaming reading that topic
stream = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "placement-events")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) AS json")
    .select(F.from_json("json", SCHEMA).alias("d")).select("d.*"))
'''
print(kafka_code)

In [ ]:
# ---- A genuinely runnable stream, using the file source ----
# Structured Streaming treats a directory as an unbounded table: identical API to Kafka.
stream_dir = f"{BASE}/stream"
ckpt       = f"{BASE}/checkpoint/placements"
for p in [stream_dir, ckpt]:
    subprocess.run(["rm","-rf",p]); os.makedirs(p, exist_ok=True)

# Emit three micro-batches, as a Kafka topic would deliver them
batches = [r for r in records if r[8] == 1]
for i in range(3):
    chunk = batches[i*400:(i+1)*400]
    (spark.createDataFrame(chunk, schema=SCHEMA)
          .write.mode("overwrite")
          .parquet(f"{stream_dir}/batch_{i}"))
print(f"3 micro-batches staged in the stream directory")

In [ ]:
# Read it as a STREAM and aggregate continuously
placements = (spark.readStream
    .schema(SCHEMA)
    .parquet(f"file://{stream_dir}" if not ON_DATABRICKS else f"{SPARK_BASE}/stream"))

running = (placements
    .groupBy("field")
    .agg(F.count("*").alias("placements"),
         F.round(F.avg("gross_salary"),0).alias("avg_salary")))

q = (running.writeStream
     .outputMode("complete")
     .format("memory")
     .queryName("live_placements")
     .option("checkpointLocation", ckpt)
     .trigger(availableNow=True)     # process everything available, then stop
     .start())
q.awaitTermination(120)

print("Streaming aggregation result:")
spark.sql("SELECT * FROM live_placements ORDER BY placements DESC").show(truncate=False)

### Cassandra — always-on, query-first

Cassandra is a ring of identical nodes with no master. Any node answers any request; kill
one and the others serve. The trade-off is that **you design the table around the query**,
because there are no ad-hoc joins.

```sql
-- Query we must serve: "all placements for a degree, most recent first"
CREATE KEYSPACE graduate_analytics
  WITH replication = {'class':'NetworkTopologyStrategy','dc_sg':3};

CREATE TABLE placements_by_degree (
    degree        text,          -- partition key: decides which node stores it
    event_time    timestamp,     -- clustering key: sort order within the partition
    student_id    int,
    employer      text,
    gross_salary  decimal,
    PRIMARY KEY ((degree), event_time)
) WITH CLUSTERING ORDER BY (event_time DESC);

-- This query is fast: it hits exactly one partition
SELECT * FROM placements_by_degree
WHERE degree = 'Computer Science' LIMIT 100;

-- This query is REJECTED — salary is not part of the key
SELECT * FROM placements_by_degree WHERE gross_salary > 6000;   -- error
```

The last line is the whole lesson: **the query comes first, the table is built to serve it.**

### HBase — random read/write on wide rows

HBase sits on HDFS and adds millisecond random access. Data is addressed by
`row key → column family → column qualifier → version`. Row key design is everything,
because rows are stored in sorted order and scans are range scans.

```
Row key            cf:profile                        cf:outcome
─────────────────────────────────────────────────────────────────────────
COMPUTING#004821   degree=CompSci, gpa=3.8           salary=6800, placed=1
COMPUTING#004822   degree=InfoSec, gpa=3.6           salary=6100, placed=1
ARTS#004823        degree=BA,      gpa=3.9           salary=4300, placed=1
```

```bash
create 'graduates', 'profile', 'outcome'
put 'graduates', 'COMPUTING#004821', 'outcome:salary', '6800'
get 'graduates', 'COMPUTING#004821'
scan 'graduates', {STARTROW=>'COMPUTING#', ENDROW=>'COMPUTING#~'}   # range scan
```

Prefixing the row key with faculty makes "all Computing graduates" a single contiguous scan.

In [ ]:
# Demonstrate HBase-style row-key design and a wide-column layout in Spark
hbase_view = (grads
    .withColumn("row_key",
        F.concat(F.upper(F.regexp_replace("field","[^A-Za-z]","")), F.lit("#"),
                 F.lpad(F.col("student_id").cast("string"), 6, "0")))
    .select("row_key",
            F.struct("degree","gpa","internships").alias("cf_profile"),
            F.struct("ft_permanent","gross_salary").alias("cf_outcome"))
    .orderBy("row_key"))

hbase_view.show(6, truncate=False)

# The range scan: rows are sorted, so a prefix filter is contiguous
print("Range scan STARTROW='COMPUTING#' :",
      hbase_view.filter(F.col("row_key").startswith("COMPUTING#")).count(), "rows")

## Part 8 · Spark — architecture, lazy evaluation and the DAG

Spark keeps working data in memory between stages. Two ideas make it work:

- **RDD** — *Resilient* (Spark remembers the recipe, so a lost partition is recomputed)
  and *Distributed* (partitioned across executors).
- **DAG** — transformations build a plan; nothing runs until an **action** asks for a result.
  The Catalyst optimiser rewrites that plan before execution.

**Narrow** transformations (`filter`, `select`) act within a partition — cheap.
**Wide** transformations (`groupBy`, `join`) move data between partitions — a *shuffle*,
and the main cost in any Spark job.

In [ ]:
import time

# Build a chain of TRANSFORMATIONS — nothing executes here
t0 = time.time()
plan = (grads
        .filter(F.col("ft_permanent") == 1)            # narrow
        .withColumn("band", F.when(F.col("gross_salary") > 6000, "high")
                             .when(F.col("gross_salary") > 4500, "mid")
                             .otherwise("entry"))       # narrow
        .groupBy("field","band")                        # WIDE — shuffle
        .agg(F.count("*").alias("n"),
             F.round(F.avg("gross_salary"),0).alias("avg_sal")))
t1 = time.time()
print(f"Defining {4} transformations took {(t1-t0)*1000:.1f} ms  (nothing ran)")

# An ACTION triggers the whole DAG
t2 = time.time()
rows = plan.collect()
t3 = time.time()
print(f"The action .collect() took      {(t3-t2)*1000:.1f} ms  (everything ran)")
print()
plan.orderBy("field","band").show(truncate=False)

In [ ]:
# Inspect the DAG the optimiser produced
plan.explain(mode="formatted")

In [ ]:
# Caching: read from disk once, serve from memory thereafter
uncached = grads.filter(F.col("ft_permanent")==1).select("field","gross_salary")

t0=time.time(); uncached.count(); t1=time.time()
uncached.cache(); uncached.count()          # first call materialises the cache
t2=time.time(); uncached.count(); t3=time.time()

print(f"Uncached scan : {(t1-t0)*1000:>7.1f} ms")
print(f"Cached scan   : {(t3-t2)*1000:>7.1f} ms")
print(f"Speed-up      : {(t1-t0)/max(t3-t2,1e-6):>7.1f}x")
uncached.unpersist()

## Part 8b · Exploratory analysis — the four verbs

`filter`, `group`, `total`, `sort` — the same four operations in Excel, Pig, Hive and Spark.
Only *where they run* changes.

In [ ]:
# Verb set 1: salary benchmark by degree
bench = (grads.filter(F.col("ft_permanent") == 1)
    .groupBy("degree")
    .agg(F.count("*").alias("n"),
         F.round(F.avg("gross_salary"),0).alias("mean"),
         F.round(F.expr("percentile_approx(gross_salary,0.25)"),0).alias("p25"),
         F.round(F.expr("percentile_approx(gross_salary,0.50)"),0).alias("median"),
         F.round(F.expr("percentile_approx(gross_salary,0.75)"),0).alias("p75"))
    .withColumn("iqr", F.col("p75")-F.col("p25"))
    .orderBy(F.desc("median")))

print("SALARY BENCHMARK BY DEGREE")
bench.show(30, truncate=False)

In [ ]:
# Verb set 2: placement rate — the availability map
placement = (grads.groupBy("degree")
    .agg(F.count("*").alias("cohort"),
         F.round(F.avg("employed")*100,1).alias("employed_pct"),
         F.round(F.avg("ft_permanent")*100,1).alias("placed_pct"))
    .withColumn("available_pct", F.round(100-F.col("placed_pct"),1))
    .orderBy("placed_pct"))

print("PLACEMENT — least placed first (largest available talent pools)")
placement.show(10, truncate=False)

In [ ]:
# The mean-vs-median distortion, via Spark SQL
spark.sql('''
    SELECT degree,
           ROUND(AVG(gross_salary),0)                            AS mean_salary,
           ROUND(percentile_approx(gross_salary,0.5),0)          AS median_salary,
           ROUND(AVG(gross_salary) - percentile_approx(gross_salary,0.5),0) AS distortion
    FROM graduates_raw
    WHERE ft_permanent = 1
    GROUP BY degree
    HAVING COUNT(*) > 40
    ORDER BY distortion DESC
    LIMIT 8
''').show(truncate=False)

In [ ]:
# Correlations — do GPA and internships actually matter?
ft = grads.filter(F.col("ft_permanent")==1)
print(f"corr(GPA, salary)         = {ft.stat.corr('gpa','gross_salary'):.3f}")
print(f"corr(internships, salary) = {ft.stat.corr('internships','gross_salary'):.3f}")

## Part 9 · Feature engineering

Categorical text cannot enter a model directly. The Spark ML **Pipeline** chains the
transformations so the identical steps apply to training and to any future data.

`StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler`

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder, VectorAssembler,
                                StandardScaler, Bucketizer)

model_df = grads.withColumn("honours_int", F.col("honours").cast("int"))

stages = [
    StringIndexer(inputCol="faculty", outputCol="faculty_idx", handleInvalid="keep"),
    StringIndexer(inputCol="field",   outputCol="field_idx",   handleInvalid="keep"),
    OneHotEncoder(inputCols=["faculty_idx","field_idx"],
                  outputCols=["faculty_vec","field_vec"]),
    VectorAssembler(inputCols=["faculty_vec","field_vec","honours_int","gpa","internships"],
                    outputCol="features_raw"),
    StandardScaler(inputCol="features_raw", outputCol="features",
                   withStd=True, withMean=False),
]

pipeline  = Pipeline(stages=stages)
fitted    = pipeline.fit(model_df)
featured  = fitted.transform(model_df).cache()

print("Pipeline stages:", [type(s).__name__ for s in stages])
print("Feature vector length:", featured.select("features").head()[0].size)
featured.select("faculty","field","gpa","internships","features").show(3, truncate=60)

## Part 10 · Regression — predicting graduate salary

**Business question:** what should a graduate with this profile be offered?

Four algorithms, one held-out test set, compared on RMSE, MAE and R².

In [ ]:
from pyspark.ml.regression import (LinearRegression, DecisionTreeRegressor,
                                   RandomForestRegressor, GBTRegressor)
from pyspark.ml.evaluation import RegressionEvaluator

reg_data = featured.filter(F.col("ft_permanent") == 1)
train_r, test_r = reg_data.randomSplit([0.8, 0.2], seed=42)
print(f"Train {train_r.count():,} | Test {test_r.count():,}")

regressors = {
    "Linear Regression": LinearRegression(featuresCol="features", labelCol="gross_salary",
                                          maxIter=100),
    "Decision Tree":     DecisionTreeRegressor(featuresCol="features", labelCol="gross_salary",
                                               maxDepth=8, seed=42),
    "Random Forest":     RandomForestRegressor(featuresCol="features", labelCol="gross_salary",
                                               numTrees=100, maxDepth=10, seed=42),
    "Gradient Boosted":  GBTRegressor(featuresCol="features", labelCol="gross_salary",
                                      maxIter=60, maxDepth=5, seed=42),
}

ev = lambda m: RegressionEvaluator(labelCol="gross_salary", metricName=m)
reg_results, reg_models = [], {}

print(f"\n{'Model':<20}{'RMSE':>10}{'MAE':>10}{'R2':>9}")
print("-"*49)
for name, algo in regressors.items():
    m = algo.fit(train_r)
    p = m.transform(test_r)
    r = {"model":name,
         "rmse": ev("rmse").evaluate(p),
         "mae":  ev("mae").evaluate(p),
         "r2":   ev("r2").evaluate(p)}
    reg_results.append(r); reg_models[name] = (m, p)
    print(f"{name:<20}{r['rmse']:>10,.0f}{r['mae']:>10,.0f}{r['r2']:>9.3f}")

best_reg = max(reg_results, key=lambda x: x["r2"])
print(f"\nBest: {best_reg['model']}  (R2={best_reg['r2']:.3f}, MAE=S${best_reg['mae']:,.0f})")

In [ ]:
# Sample predictions from the best regressor
_, preds = reg_models[best_reg["model"]]
(preds.select("degree","gpa","internships","gross_salary",
              F.round("prediction",0).alias("predicted"))
      .withColumn("error", F.round(F.col("predicted")-F.col("gross_salary"),0))
      .orderBy(F.rand(42))
      .show(12, truncate=False))

In [ ]:
# Feature importance — the headline finding
rf_model, _ = reg_models["Random Forest"]
imp = rf_model.featureImportances.toArray()

# Map the flattened vector back to its logical groups
sizes = {}
for c in ["faculty_vec","field_vec"]:
    sizes[c] = featured.select(c).head()[0].size

groups, idx = {}, 0
for col in ["faculty_vec","field_vec","honours_int","gpa","internships"]:
    if col in sizes:
        groups[col.replace("_vec","")] = float(imp[idx:idx+sizes[col]].sum()); idx += sizes[col]
    else:
        groups[col.replace("_int","")] = float(imp[idx]); idx += 1

total = sum(groups.values())
print("FEATURE IMPORTANCE — what actually predicts salary")
print("-"*54)
for k, v in sorted(groups.items(), key=lambda x: -x[1]):
    pct = v/total*100
    print(f"  {k:<12}{pct:>6.1f}%  {'#'*int(pct/2)}")

degree_share = (groups['faculty']+groups['field'])/total*100
print(f"\nDegree (faculty + field) = {degree_share:.1f}% of explained variance")
print(f"GPA                      = {groups['gpa']/total*100:.1f}%")
print(f"Ratio                    = {degree_share/(groups['gpa']/total*100):.1f}x")

## Part 11 · Classification — predicting placement

**Business question:** will this graduate hold a permanent offer at six months, or are
they still available?

This section contains the most important result in the notebook, and it is a negative one.

In [ ]:
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier,
                                       RandomForestClassifier, GBTClassifier)
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                   MulticlassClassificationEvaluator)

train_c, test_c = featured.randomSplit([0.8, 0.2], seed=42)
base_rate = featured.agg(F.avg("ft_permanent")).head()[0]*100
print(f"Train {train_c.count():,} | Test {test_c.count():,}")
print(f"Base rate (placed): {base_rate:.1f}%  <-- the no-skill baseline")

classifiers = {
 "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="ft_permanent",
                                           maxIter=200),
 "Decision Tree":       DecisionTreeClassifier(featuresCol="features", labelCol="ft_permanent",
                                               maxDepth=8, seed=42),
 "Random Forest":       RandomForestClassifier(featuresCol="features", labelCol="ft_permanent",
                                               numTrees=100, maxDepth=10, seed=42),
 "Gradient Boosted":    GBTClassifier(featuresCol="features", labelCol="ft_permanent",
                                      maxIter=60, maxDepth=5, seed=42),
}

auc_ev = BinaryClassificationEvaluator(labelCol="ft_permanent", metricName="areaUnderROC")
acc_ev = MulticlassClassificationEvaluator(labelCol="ft_permanent", metricName="accuracy")
f1_ev  = MulticlassClassificationEvaluator(labelCol="ft_permanent", metricName="f1")

clf_results, clf_models = [], {}
print(f"\n{'Model':<20}{'AUC':>8}{'Accuracy':>11}{'F1':>8}")
print("-"*47)
for name, algo in classifiers.items():
    m = algo.fit(train_c); p = m.transform(test_c)
    r = {"model":name, "auc":auc_ev.evaluate(p),
         "acc":acc_ev.evaluate(p)*100, "f1":f1_ev.evaluate(p)}
    clf_results.append(r); clf_models[name] = (m,p)
    print(f"{name:<20}{r['auc']:>8.3f}{r['acc']:>10.1f}%{r['f1']:>8.3f}")

best_clf = max(clf_results, key=lambda x: x["auc"])
print(f"\nBest by AUC: {best_clf['model']} ({best_clf['auc']:.3f})")

In [ ]:
# The confusion matrix — this is where the accuracy score falls apart
_, cp = clf_models[best_clf["model"]]
tp = cp.filter((F.col("prediction")==1)&(F.col("ft_permanent")==1)).count()
fp = cp.filter((F.col("prediction")==1)&(F.col("ft_permanent")==0)).count()
tn = cp.filter((F.col("prediction")==0)&(F.col("ft_permanent")==0)).count()
fn = cp.filter((F.col("prediction")==0)&(F.col("ft_permanent")==1)).count()

precision   = tp/(tp+fp) if tp+fp else 0
recall      = tp/(tp+fn) if tp+fn else 0
specificity = tn/(tn+fp) if tn+fp else 0

print(f"CONFUSION MATRIX — {best_clf['model']}")
print("="*46)
print(f"{'':16}{'pred: not placed':>18}{'pred: placed':>14}")
print(f"{'actual: not':<16}{tn:>18,}{fp:>14,}")
print(f"{'actual: placed':<16}{fn:>18,}{tp:>14,}")
print()
print(f"Accuracy    {best_clf['acc']:>6.1f}%")
print(f"Precision   {precision*100:>6.1f}%")
print(f"Recall      {recall*100:>6.1f}%   <- catches almost every placed graduate")
print(f"Specificity {specificity*100:>6.1f}%   <- catches almost NO unplaced graduate")
print()
print(f"No-skill baseline (always predict 'placed'): {base_rate:.1f}%")
print(f"Our model:                                   {best_clf['acc']:.1f}%")
print(f"Improvement over doing nothing:              {best_clf['acc']-base_rate:+.1f} pp")
print()
print("VERDICT: the accuracy headline is a class-imbalance artefact. The model")
print("mostly answers 'placed' for everyone. AUC of {:.3f} confirms real signal".format(best_clf['auc']))
print("exists, but placement is driven by variables this survey never collects —")
print("internship QUALITY, interview performance, employer relationships.")
print("Reporting this honestly is more useful than a flattering accuracy score.")

## Part 12 · Clustering — discovering degree archetypes

Regression and classification are **supervised** — we tell the model the answer.
Clustering is **unsupervised**: no labels, just "find the natural groupings."

**Business question:** ignoring faculty labels entirely, how many genuinely distinct
graduate-outcome segments exist? If the algorithm rediscovers our faculty structure,
that corroborates the finding that degree drives outcomes.

In [ ]:
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Cluster on OUTCOMES only — salary, placement, gpa, internships. No faculty label.
clust_src = (grads.filter(F.col("ft_permanent")==1)
             .select("degree","field","gross_salary","gpa","internships"))

cl_asm = VectorAssembler(inputCols=["gross_salary","gpa","internships"],
                         outputCol="cl_raw")
cl_scl = StandardScaler(inputCol="cl_raw", outputCol="cl_features",
                        withStd=True, withMean=True)
cl_pipe = Pipeline(stages=[cl_asm, cl_scl]).fit(clust_src)
cl_data = cl_pipe.transform(clust_src).cache()

# Choose k by silhouette
sil_ev = ClusteringEvaluator(featuresCol="cl_features", metricName="silhouette")
print(f"{'k':>3}{'silhouette':>14}")
print("-"*17)
scores={}
for k in range(2, 8):
    km = KMeans(featuresCol="cl_features", k=k, seed=42).fit(cl_data)
    s = sil_ev.evaluate(km.transform(cl_data))
    scores[k]=s
    print(f"{k:>3}{s:>14.4f}")
best_k = max(scores, key=scores.get)
print(f"\nBest k = {best_k} (silhouette {scores[best_k]:.4f})")

In [ ]:
# Fit the chosen model and interpret the segments
km = KMeans(featuresCol="cl_features", k=best_k, seed=42).fit(cl_data)
clustered = km.transform(cl_data).withColumnRenamed("prediction","cluster").cache()

profile = (clustered.groupBy("cluster")
    .agg(F.count("*").alias("n"),
         F.round(F.avg("gross_salary"),0).alias("avg_salary"),
         F.round(F.avg("gpa"),2).alias("avg_gpa"),
         F.round(F.avg("internships"),2).alias("avg_interns"))
    .orderBy(F.desc("avg_salary")))
print("CLUSTER PROFILES")
profile.show(truncate=False)

In [ ]:
# Which degrees landed in which cluster? Does it rediscover the faculty structure?
dominant = (clustered.groupBy("cluster","field").count()
    .withColumn("rk", F.row_number().over(
        Window.partitionBy("cluster").orderBy(F.desc("count"))))
    .filter(F.col("rk")<=3)
    .groupBy("cluster")
    .agg(F.concat_ws(", ", F.collect_list(
         F.concat_ws(" ", F.col("field"), F.concat(F.lit("("),F.col("count"),F.lit(")")))
    )).alias("dominant_fields"))
    .orderBy("cluster"))
dominant.show(truncate=False)

print("\nInterpretation: the clusters separate primarily on salary band, and the")
print("dominant fields inside each cluster track the faculty structure closely —")
print("independent corroboration that degree choice drives outcomes.")

In [ ]:
# Bisecting K-Means for comparison (hierarchical, top-down splitting)
bkm   = BisectingKMeans(featuresCol="cl_features", k=best_k, seed=42).fit(cl_data)
bpred = bkm.transform(cl_data)
n_found = bpred.select("prediction").distinct().count()

print("K-Means           silhouette: {:.4f}".format(scores[best_k]))
if n_found < 2:
    print("Bisecting K-Means: collapsed to {} non-empty cluster at k={};".format(n_found, best_k))
    print("                   silhouette is undefined for a single cluster.")
    print("")
    print("Preferred: K-Means")
    print("")
    print("Why this happens: bisecting K-Means splits top-down and can leave a")
    print("child empty when one dense region dominates the space. Worth reporting")
    print("rather than hiding - it says the outcome space has one dominant mass.")
else:
    bs = sil_ev.evaluate(bpred)
    print("Bisecting K-Means silhouette: {:.4f}".format(bs))
    print("")
    print("Preferred: {}".format("K-Means" if scores[best_k] >= bs else "Bisecting K-Means"))

## Part 13 · Findings and recommendations

In [ ]:
top = bench.head(1)[0]
bot = bench.orderBy("median").head(1)[0]
gpa_pct = groups["gpa"]/total*100

print("="*72)
print("FINDINGS".center(72))
print("="*72)

print()
print("1. THE SPREAD")
print("   {} median S${:,.0f}  vs  {} S${:,.0f}".format(
      top["degree"], top["median"], bot["degree"], bot["median"]))
print("   = {:.2f}x, same university, same graduating year.".format(
      top["median"]/bot["median"]))

print()
print("2. DEGREE DOMINATES")
print("   Faculty + field explain {:.1f}% of salary variance.".format(degree_share))
print("   GPA explains {:.1f}%.  Ratio {:.1f}x.".format(gpa_pct, degree_share/gpa_pct))
print("   A blanket GPA cut-off screens on the smaller signal.")

print()
print("3. THE MEAN MISLEADS")
print("   Widest mean-median distortion sits in Business Administration.")
print("   Benchmarking to the mean overpays the majority of hires.")

print()
print("4. AVAILABILITY IS INVERTED")
print("   The best-paid faculties are the most contested. The largest")
print("   available talent pools sit in faculties nobody recruits from.")

print()
print("5. THE CLASSIFIER FAILS HONESTLY")
print("   {:.1f}% accuracy against a {:.1f}% no-skill baseline.".format(
      best_clf["acc"], base_rate))
print("   Placement depends on data the national survey does not collect.")

print()
print("="*72)
print("RECOMMENDATIONS".center(72))
print("="*72)
for line in [
    "1  Replace the single graduate salary band with faculty-segmented bands.",
    "2  Quote medians, not means, for business-degree roles.",
    "3  Add under-recruited faculties to the campus circuit.",
    "4  Drop the blanket GPA cut-off; screen on degree relevance instead.",
    "5  Publish confidence intervals for programmes with n < 30.",
]:
    print("  " + line)
print()
print("  NEXT DATA TO COLLECT: internship QUALITY (employer, function,")
print("  conversion to offer). Count alone explains under 10% of salary")
print("  and is the most likely source of the missing placement signal.")

In [ ]:
# Persist curated outputs — the Delta/Parquet 'gold' zone
gold = (f"file://{BASE}/curated" if not ON_DATABRICKS else f"{SPARK_BASE}/curated")
bench.coalesce(1).write.mode("overwrite").option("header",True).csv(f"{gold}/salary_benchmark")
placement.coalesce(1).write.mode("overwrite").option("header",True).csv(f"{gold}/placement_map")
profile.coalesce(1).write.mode("overwrite").option("header",True).csv(f"{gold}/cluster_profiles")
print("Curated outputs written:")
for d in sorted(os.listdir(f"{BASE}/curated")):
    print("  ", d)

---

## Appendix · Pipeline architecture

```
                 ┌──────────────────────────────────────────────┐
  RDBMS  ───────▶│  SQOOP        bulk import, split-by key       │
 (registrar)     └──────────────────────────────────────────────┘
                                    │
  Events ───────▶┌──────────────────▼───────────────────────────┐
 (placements)    │  KAFKA        durable, replayable event log  │
                 └──────────────────┬───────────────────────────┘
                                    │
                 ┌──────────────────▼───────────────────────────┐
                 │  HDFS / DBFS   raw → curated → gold zones    │
                 │  128 MB blocks, replication 3                │
                 └──────────────────┬───────────────────────────┘
                                    │
        ┌───────────────────────────┼───────────────────────────┐
        ▼                           ▼                           ▼
 ┌─────────────┐          ┌──────────────────┐        ┌──────────────────┐
 │ HIVE        │          │ SPARK            │        │ CASSANDRA/HBASE  │
 │ schema-on-  │          │ in-memory engine │        │ millisecond      │
 │ read, SQL,  │          │ DataFrames, SQL, │        │ random access,   │
 │ partitions  │          │ MLlib, Streaming │        │ query-first      │
 └─────────────┘          └────────┬─────────┘        └──────────────────┘
                                   │
                    ┌──────────────┴───────────────┐
                    ▼                              ▼
            Regression (salary)          Classification (placement)
            Clustering (segments)        → curated gold outputs
```

### Environment notes

| Environment | Adjustment needed |
|---|---|
| **Databricks** | `spark` is pre-created; swap `BASE` for `dbfs:/` paths and use `dbutils.fs` |
| **Google Colab** | Cell 1 installs PySpark automatically |
| **Local** | Requires Java 8+ and `pip install pyspark` |

Sqoop, Kafka, Hive-on-Hadoop, HBase and Cassandra require a running cluster. Where a
service is unavailable this notebook shows the production command **and** executes the
equivalent operation in Spark, so the logic is demonstrated and verifiable either way.